# 10 · Labels, features and walk-forward splits

Builds and checks the inputs every model uses (tasks 2.4, 10.1, 16.1, 16.2):

1. **Adjusted prices**: NSE prices adjusted for splits, bonuses, rights issues and demergers.
2. **Features**: 39 inputs per stock and trading day, each known at the 08:30 IST forecast.
3. **Labels**: direction and realised volatility over 1, 5 and 10 trading days.
4. **Walk-forward splits**: the gap rule that keeps training labels from reaching into a prediction period.

Return and signal analysis here uses train and validation (2016–2023) only.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.analysis.plots import (BLUE_CMAP, HORIZON_COLORS, MODEL_COLORS, TEXT_2, apply_style, bar_style,
                                dot_compare, figures_dir, save)
from src.config import Config
from src.evaluation.metrics import calibration_table, evaluate, join_labels
from src.models.data import label_frame, load_panel
from src.models.walkforward import run_or_load

cfg = Config.load()
apply_style()
pd.set_option("display.width", 160)
pd.set_option("display.precision", 4)
val_start, val_end = cfg.period("validation")

In [ ]:
REBUILD = False          # True rebuilds adjusted prices, features and labels from data/lake
if REBUILD or not (cfg.derived_dir / "labels" / "labels.parquet").exists():
    from src.models.data import prepare
    prepare(cfg)
panel = load_panel(cfg)
_, train_end = cfg.period("train")
period = np.select([panel.df.trade_date <= pd.Timestamp(train_end), panel.df.trade_date <= pd.Timestamp(val_end)],
                   ["train", "validation"], "test")
df = panel.df.assign(period=period)
ready = df[df.ready]

## 1. Corporate-action adjustment

An event's factor multiplies every earlier price (volume is divided by it), so the series has no jump on the ex-date.
Split and bonus factors come from NSE's subject text. The other events are listed below for the hand check task 2.4 asks for;
`data/manual/ca_adjustments.csv` overrides any of them.

In [ ]:
events = pd.read_parquet(cfg.derived_dir / "ca_events" / "ca_events.parquet")
print(events.groupby("method").size().rename("events").to_string())
events.loc[events.method != "subject_ratio", ["symbol", "ex_date", "kind", "factor", "method", "detail"]]

In [ ]:
jumps = pd.read_parquet(cfg.derived_dir / "jump_report" / "jump_report.parquet")
print(f"Adjusted daily moves above 20%: {len(jumps)}; on a corporate-action day: {int(jumps.on_event_day.sum())}")
jumps

## 2. The panel

In [ ]:
summary = ready.groupby("period").agg(rows=("symbol", "size"), stocks=("symbol", "nunique"),
                                      first=("trade_date", "min"), last=("trade_date", "max"))
print(f"{len(df):,} stock-days for {df.symbol.nunique()} stocks; {len(ready):,} usable "
      f"(at least {cfg['features']['min_history']} trading days of history)")
summary.loc[["train", "validation", "test"]]

## 3. Labels

Labels start at the open of day d (`labels.entry: open` in `config.yaml`). The share of up moves is the accuracy a
constant guess gets, so it is the first bar every model has to clear.

In [ ]:
rows = []
for name, g in ready[ready.period != "test"].groupby("period"):
    for h in panel.horizons:
        rows.append({"period": name, "horizon": h, "up_share": g[f"up_{h}"].mean(),
                     "mean_log_return": g[f"fwd_ret_{h}"].mean(), "realised_vol": g[f"fwd_vol_{h}"].mean()})
base_rates = pd.DataFrame(rows).pivot(index="horizon", columns="period")
base_rates

In [ ]:
hist = ready[ready.period != "test"]
by_year = hist.groupby(hist.trade_date.dt.year)[[f"up_{h}" for h in panel.horizons]].mean()
fig, ax = plt.subplots(figsize=(8, 3.6))
for h in panel.horizons:
    ax.plot(by_year.index, by_year[f"up_{h}"], marker="o", color=HORIZON_COLORS[h], label=f"{h}-day")
    ax.annotate(f"{h}-day", (by_year.index[-1], by_year[f"up_{h}"].iloc[-1]), xytext=(8, 0),
                textcoords="offset points", va="center", color=TEXT_2, fontsize=9)
ax.axhline(0.5, color=TEXT_2, linewidth=1, linestyle="--")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylabel("share of up moves")
ax.set_title("Share of up moves from the open, by year (Nifty 50 stocks)")
ax.legend(loc="upper center", ncols=3)
save(fig, cfg, "10_up_share_by_year")
plt.show()
by_year.round(3)

## 4. Features

All 39 features are as of 08:30 IST on the forecast day. The chart ranks them by their Spearman correlation with
the next 5-day return on the training rows, pooled across stocks. Single-feature links this small are normal for
daily stock returns; the models combine many weak inputs.

In [ ]:
train = ready[ready.period == "train"]
train[panel.features].describe().T[["mean", "std", "min", "max"]]

In [ ]:
rho = train[panel.features].corrwith(train["fwd_ret_5"], method="spearman")
top = rho.reindex(rho.abs().sort_values(ascending=False).index[:15]).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.6))
ax.barh(top.index, top.values, color=BLUE_CMAP(0.55), height=0.6, **bar_style())
ax.axvline(0, color=TEXT_2, linewidth=1)
ax.set_xlabel("Spearman correlation with the 5-day forward return (train, 2016-2022)")
ax.set_title("Strongest single-feature links to the 5-day return")
ax.grid(axis="y", visible=False)
save(fig, cfg, "10_feature_rank_correlation")
plt.show()
rho.sort_values(key=abs, ascending=False).round(4).to_frame("spearman_5d")

## 5. Walk-forward splits

A training row for day t looks at prices up to day t+h−1. For a fold that starts predicting on day T, a row trains
only if that end date, for the 10-day horizon, is before T. ARIMA refits monthly; the other models refit yearly.

In [ ]:
from src.labels.splits import make_folds, train_mask

rows = []
for f in make_folds(panel.dates, val_start, val_end, "monthly"):
    m = train_mask(panel.df, f.start, panel.max_h)
    rows.append({"predict_from": f.start, "predict_to": f.end,
                 "last_train_day": panel.df.loc[m, "trade_date"].max(),
                 "last_label_end": pd.to_datetime(panel.df.loc[m, "end_10"]).max(), "train_rows": int(m.sum())})
splits = pd.DataFrame(rows)
assert (splits.last_label_end < splits.predict_from).all(), "gap rule broken"
splits

In [ ]:
t0 = pd.Timestamp("2022-10-01")
fig, ax = plt.subplots(figsize=(8, 4))
for i, r in splits.iterrows():
    ax.barh(i, (r.last_train_day - t0).days, left=mdates.date2num(t0), height=0.6, color=HORIZON_COLORS[1],
            label="training rows" if i == 0 else None, **bar_style())
    ax.barh(i, (r.predict_to - r.predict_from).days + 1, left=mdates.date2num(r.predict_from), height=0.6,
            color=MODEL_COLORS["lstm"], label="predicted month" if i == 0 else None, **bar_style())
ax.set_yticks(range(len(splits)), [d.strftime("%b") for d in splits.predict_from])
ax.invert_yaxis()
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.set_xlim(mdates.date2num(t0), mdates.date2num(pd.Timestamp("2024-01-01")))
ax.grid(axis="y", visible=False)
ax.set_title("Monthly walk-forward folds in 2023 (training starts in Oct 2016)")
ax.legend(loc="upper right")
save(fig, cfg, "10_walk_forward_folds")
plt.show()

## Findings

1. **Adjustment:** 49 corporate-action events: 33 splits/bonuses read from NSE's text, 7 rights issues (theoretical ex-rights price), 7 demergers (ex-date open / previous close), 1 entered by hand (the JSW Steel split of 4 Jan 2017, missing from NSE's list) and 1 without prices (SHRIRAMFIN's 2020 rights issue, before its symbol's history). After adjustment, none of the 14 daily moves above 20% falls on a corporate-action day; they are market events such as 23 Mar 2020 and the Adani sell-off of Feb 2023.
2. **Panel:** 125,999 stock-days for 50 stocks; 115,999 usable from 26 Oct 2016 (200 days of history). Train 68,906 rows (48 stocks), validation 11,863 (49), test 35,230 (50).
3. **Base rates (from the open):** training 45.8% / 51.3% / 53.2% up at 1 / 5 / 10 days; validation 2023 46.9% / 56.2% / 60.6%. The 1-day open-to-close move is down more often than up in every year (44-48%): the drift arrives overnight, before the entry. 2023 has the highest 5- and 10-day up-shares of any year, so a constant 'up' guess is hard to beat in validation.
4. **Features:** every single-feature link to the 5-day return is weak; the largest Spearman correlation is 0.046 (India VIX level). Trend features have negative links (EMA20/EMA50 -0.023, distance from the 52-week high -0.029): over 5 days these stocks mean-reverted in 2016-2022.
5. **Gap rule:** each monthly fold trains only on rows whose 10-day label ends before it starts; e.g. the January 2023 fold trains on rows up to 19 Dec 2022, whose labels end on 30 Dec 2022.